# Báo Cáo Cuối Kỳ - K-Means Image Segmentation

## 1. Define Problem

Bài toán yêu cầu phân đoạn một ảnh thành `K` cụm bằng thuật toán K-Means. Trong chương trình này, ảnh đầu vào là ảnh phong cảnh vì các vùng như bầu trời, nước, cây, núi, cát và mây thường có màu sắc tách biệt rõ, phù hợp với phân cụm màu không giám sát.

Notebook này là bản báo cáo cuối cùng, chỉ dùng Markdown và hình ảnh, có **0 code cell**. Toàn bộ code train, so sánh mô hình, xuất ảnh, xuất nhãn pixel, lưu model artifact và review hệ thống đã được chạy trước khi tạo notebook.

## 2. Assignment Mapping

| Yêu cầu đề bài | Phần đã thực hiện |
|---|---|
| Load the image | Đọc ảnh RGB bằng PIL trong `src/image_io.py` |
| Convert color space | So sánh `RGB`, `HSV`, `LAB` |
| Resize image | Resize với `max_side=128` để train nhanh và ổn định |
| Flatten pixels | Biến ảnh thành ma trận `(height * width, n_features)` |
| Apply K-Means | Cài đặt K-Means from scratch bằng NumPy |
| Assign nearest centroid | Dùng Euclidean distance dạng vector hóa |
| Update centroids | Cập nhật centroid bằng mean của pixel trong cụm |
| Repeat until convergence | Dừng bằng `max_iter` và `tol` |
| Segment image | Thay mỗi pixel bằng màu centroid |
| Visualize results | Xuất ảnh segmented, comparison và K-grid |

## 3. Workflow

Workflow đã thực hiện:

1. **Thu thập data thô**: ảnh nằm trong `../data/raw/api` và manifest `../data/manifest/raw_landscape_manifest.csv`.
2. **Kiểm tra và cân bằng data**: clean manifest `../data/manifest/clean_landscape_manifest.csv` và báo cáo `../reports/metrics/data_balance_report.json`.
3. **Tiền xử lý**: đọc ảnh RGB, resize, đổi color space, flatten pixel, tùy chọn thêm tọa độ `(x, y)`.
4. **Train K-Means**: train trên `K=2..10`, `rgb/hsv/lab`, `use_xy=false/true`.
5. **Xuất kết quả**: ảnh trong `../reports/figures`, labels trong `../data/labels`, model artifacts trong `../models`.
6. **So sánh và review**: metrics `../reports/metrics/model_comparison.csv`, best model `../reports/metrics/best_model_by_image.json`, review `../reports/metrics/source_review.json`.

## 4. Data Audit

Hệ thống kiểm tra các đặc trưng của từng ảnh: width, height, aspect ratio, mean intensity, standard deviation, file size, source và query. Các ảnh lệch quá xa phân phối được đánh dấu bằng IQR và z-score trước khi chọn tập balanced.

**Tổng quan data sau cân bằng**

| raw_images | clean_images | real_selected | augmentation_selected | max_augmentation | model_runs |
| --- | --- | --- | --- | --- | --- |
| 22 | 20 | 20 | 0 | 8 | 1080 |

**Khoảng giá trị sau cân bằng**

| Feature | Min | Mean | Max |
|---|---:|---:|---:|
| width | 720 | 2848.90 | 7580 |
| height | 663 | 1899.55 | 4667 |
| mean intensity | 65.87 | 107.26 | 164.67 |
| std intensity | 44.59 | 57.10 | 77.39 |

## 5. Data Balancing

Mục tiêu là giữ `20` ảnh sạch, giảm lệ thuộc augmentation và ưu tiên ảnh real/raw từ Wikimedia. Kết quả cuối cùng chọn **20 ảnh real** và **0 ảnh augmentation**.

**Phân phối source**

| source | count |
| --- | --- |
| wikimedia_commons | 4 |
| wikimedia_commons_category | 2 |
| wikimedia_existing_raw | 14 |

**Phân phối query**

| query | count |
| --- | --- |
| Desert landscapes | 1 |
| Landscape photographs | 1 |
| beach landscape | 1 |
| existing raw landscape | 14 |
| forest landscape | 1 |
| lake landscape | 1 |
| national park | 1 |

## 6. Preprocessing

Mỗi ảnh được resize về `max_side=128`, sau đó thử nghiệm trên ba không gian màu `RGB`, `HSV`, `LAB`. Mỗi pixel trở thành vector đặc trưng màu. Ở chế độ spatial, vector được mở rộng thêm tọa độ chuẩn hóa `(x, y)`.

Không dùng mask ground-truth, semantic segmentation, object detection hoặc deep learning model. Đây vẫn là bài toán K-Means Image Segmentation đúng trọng tâm đề bài.

## 7. Training Grid

| Thành phần | Giá trị |
|---|---|
| Số ảnh clean | 20 |
| K values | [2, 3, 4, 5, 6, 7, 8, 9, 10] |
| Color spaces | ['rgb', 'hsv', 'lab'] |
| Spatial modes | [False, True] |
| Tổng số model runs | 1080 |

Mỗi model run đều xuất ra ảnh segmented, ảnh comparison, pixel-label `.npy`, và model artifact `.npz`.

## 8. Model Comparison

Vì đề bài không cung cấp ground-truth segmentation mask, chất lượng model được đánh giá bằng unsupervised metrics:

| Metric | Hướng tốt hơn | Ý nghĩa |
|---|---|---|
| silhouette_sample | cao hơn | Cụm tách nhau rõ hơn |
| davies_bouldin_sample | thấp hơn | Cụm gọn và ít chồng lấn hơn |
| calinski_harabasz_sample | cao hơn | Cụm tách biệt tốt hơn |
| inertia_per_pixel | thấp hơn | Pixel gần centroid hơn |
| cluster_balance | cao hơn | Tránh cụm quá nhỏ/collapsed |
| ranking_score | thấp hơn | Điểm tổng hợp để chọn model |

**Top 15 model runs**

| image_id | k | color_space | use_xy | silhouette_sample | davies_bouldin_sample | inertia_per_pixel | cluster_balance | ranking_score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 0.7680300230392144 | 0.3889383730290883 | 3397.796426673979 | 0.9824740167108212 | 1.4515519988570056 |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | True | 0.7585101716545529 | 0.397228702572854 | 3467.828345595299 | 0.9824740167108212 | 1.4714017305432043 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 0.7768994629167776 | 0.2886640635269787 | 766.5654205675322 | 0.648 | 1.5280869720455237 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | True | 0.7543469492889678 | 0.3209496396153748 | 838.4400671829987 | 0.6494432628549981 | 1.5904329357596856 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 0.663717964580371 | 0.450641220531663 | 2001.840447158684 | 0.9514781917009149 | 1.5956772318061554 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | False | 0.6591137989567841 | 0.4640409341143357 | 1104.0615972266949 | 0.885970531710442 | 1.619722239096586 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 0.6615684530453798 | 0.4770289328753115 | 2524.9980681173874 | 0.9713716252944374 | 1.6275280083465953 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | True | 0.6525974237422562 | 0.4670227509144705 | 2090.7033874196254 | 0.9507023588656242 | 1.6282963713982146 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | True | 0.6512798551101667 | 0.4917372826275049 | 2608.9041932243795 | 0.9731592310482408 | 1.6540050756757108 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | True | 0.635919186911147 | 0.5002377159598438 | 1192.7235343260672 | 0.8852459016393442 | 1.6875558995444644 |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 0.6424904349545513 | 0.464546509915471 | 985.383811400412 | 0.7890724269377383 | 1.7223882020147208 |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 0.6614702077130843 | 0.4464239423673495 | 2420.6202740715503 | 0.814959234314073 | 1.7492137133254575 |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | 0.6463516030822222 | 0.4776364985860166 | 1262.2506603908505 | 0.7879869625950644 | 1.7574422896030486 |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | 0.7852136269873675 | 0.3137703042279379 | 2459.2277244384554 | 0.5518394648829431 | 1.7575181319128204 |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | 0.6245320484158806 | 0.5304142507342385 | 4330.663151779813 | 0.9675491033304868 | 1.7757037836846037 |

## 9. Best Model Selection

Với mỗi ảnh, model tốt nhất là cấu hình có `ranking_score` thấp nhất.

| image_id | k | color_space | use_xy | silhouette_sample | davies_bouldin_sample | ranking_score |
| --- | --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 0.7680300230392144 | 0.3889383730290883 | 1.4515519988570056 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 0.7768994629167776 | 0.2886640635269787 | 1.5280869720455237 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 0.663717964580371 | 0.450641220531663 | 1.5956772318061554 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 0.6615684530453798 | 0.4770289328753115 | 1.6275280083465953 |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 0.6424904349545513 | 0.464546509915471 | 1.7223882020147208 |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 0.6614702077130843 | 0.4464239423673495 | 1.7492137133254575 |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | 0.6463516030822222 | 0.4776364985860166 | 1.7574422896030486 |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | 0.7852136269873675 | 0.3137703042279379 | 1.7575181319128204 |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | 0.6245320484158806 | 0.5304142507342385 | 1.7757037836846037 |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | lab | False | 0.875077260570786 | 0.2084735871637714 | 1.8488095325818388 |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | lab | False | 0.8390200224812907 | 0.2337744664684616 | 1.8512513132609527 |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | lab | False | 0.8359389243991842 | 0.2633130697616206 | 1.978389245044346 |
| Castle_Mountain_jpg | 2 | lab | False | 0.6381320390338326 | 0.6136415221891759 | 2.072480937059499 |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | rgb | False | 0.5400444062908873 | 0.6752301069700718 | 2.08084331621532 |
| jpg | 2 | lab | False | 0.6729276680462352 | 0.5878205322606979 | 2.101153243700912 |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | rgb | False | 0.7549214309460072 | 0.3563897037397857 | 2.105833867868676 |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | rgb | False | 0.5537416068602463 | 0.6251626015634806 | 2.142069657872005 |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | lab | False | 0.7767430361162948 | 0.2713149258660801 | 2.1538381192350964 |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | lab | False | 0.6349555459172955 | 0.6678555841204598 | 2.273377819370455 |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | lab | False | 0.8103970887812879 | 0.3781680639510036 | 2.280080992452206 |

## 10. Hình Ảnh Model Tốt Nhất Sau Khi Train - Best Output Image Gallery

Phần này nhúng trực tiếp ảnh comparison của model tốt nhất cho từng ảnh. Đây là các hình sau khi train, gồm ảnh gốc và ảnh đã segment đặt cạnh nhau.

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=1.4515519988570056](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=1.4515519988570056*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.5280869720455237](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.5280869720455237*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.5956772318061554](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.5956772318061554*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.6275280083465953](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.6275280083465953*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=1.7223882020147208](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=1.7223882020147208*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=rgb | use_xy=False | ranking_score=1.7492137133254575](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=rgb | use_xy=False | ranking_score=1.7492137133254575*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=rgb | use_xy=False | ranking_score=1.7574422896030486](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=rgb | use_xy=False | ranking_score=1.7574422896030486*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=1.7575181319128204](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=1.7575181319128204*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.7757037836846037](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=1.7757037836846037*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=lab | use_xy=False | ranking_score=1.8488095325818388](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=lab | use_xy=False | ranking_score=1.8488095325818388*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=lab | use_xy=False | ranking_score=1.8512513132609527](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=lab | use_xy=False | ranking_score=1.8512513132609527*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=lab | use_xy=False | ranking_score=1.978389245044346](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=lab | use_xy=False | ranking_score=1.978389245044346*

![image_id=Castle_Mountain_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=2.072480937059499](../reports/figures/Castle_Mountain_jpg_k2_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=2.072480937059499*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=rgb | use_xy=False | ranking_score=2.08084331621532](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=rgb | use_xy=False | ranking_score=2.08084331621532*

![image_id=jpg | k=2 | color_space=lab | use_xy=False | ranking_score=2.101153243700912](../reports/figures/jpg_k2_lab_color_comparison.png)

*image_id=jpg | k=2 | color_space=lab | use_xy=False | ranking_score=2.101153243700912*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=2.105833867868676](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=2.105833867868676*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=2.142069657872005](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=rgb | use_xy=False | ranking_score=2.142069657872005*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=lab | use_xy=False | ranking_score=2.1538381192350964](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=lab | use_xy=False | ranking_score=2.1538381192350964*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=2.273377819370455](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=lab | use_xy=False | ranking_score=2.273377819370455*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=lab | use_xy=False | ranking_score=2.280080992452206](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=lab | use_xy=False | ranking_score=2.280080992452206*

## 11. K-Grid Highlights

Các hình K-grid dưới đây cho thấy kết quả segmentation thay đổi như thế nào khi tăng `K`.

### 11.1. K-Grid Batch 1

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png*

![Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_color.png](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_color.png)

*Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_color.png*

![Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_xy.png](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_xy.png)

*Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_hsv_xy.png*

![Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_color.png](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_color.png)

*Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_color.png*

![Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_xy.png](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_xy.png)

*Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_lab_xy.png*

![Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_color.png](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_color.png)

*Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_color.png*

![Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_xy.png](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_xy.png)

*Algoma_Gabrielle_Rock_I0012362_jpg_k_grid_rgb_xy.png*

### 11.2. K-Grid Batch 2

![Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_color.png](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_color.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_color.png*

![Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_xy.png](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_xy.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_hsv_xy.png*

![Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_color.png](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_color.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_color.png*

![Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_xy.png](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_xy.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_lab_xy.png*

![Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_color.png](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_color.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_color.png*

![Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_xy.png](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_xy.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg_k_grid_rgb_xy.png*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_color.png](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_color.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_color.png*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_xy.png](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_xy.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_hsv_xy.png*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_color.png](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_color.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_color.png*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_xy.png](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_xy.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_lab_xy.png*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_color.png](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_color.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_color.png*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_xy.png](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_xy.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k_grid_rgb_xy.png*

### 11.3. K-Grid Batch 3

![Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_hsv_color.png](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_hsv_color.png)

*Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_hsv_color.png*

![Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_hsv_xy.png](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_hsv_xy.png)

*Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_hsv_xy.png*

![Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_lab_color.png](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_lab_color.png)

*Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_lab_color.png*

![Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_lab_xy.png](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_lab_xy.png)

*Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_lab_xy.png*

![Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_rgb_color.png](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_rgb_color.png)

*Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_rgb_color.png*

![Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_rgb_xy.png](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_rgb_xy.png)

*Bontecou_Lake_Milky_Way_panorama_jpg_k_grid_rgb_xy.png*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_hsv_color.png](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_hsv_color.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_hsv_color.png*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_hsv_xy.png](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_hsv_xy.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_hsv_xy.png*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_lab_color.png](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_lab_color.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_lab_color.png*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_lab_xy.png](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_lab_xy.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_lab_xy.png*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_rgb_color.png](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_rgb_color.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_rgb_color.png*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_rgb_xy.png](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_rgb_xy.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k_grid_rgb_xy.png*

### 11.4. K-Grid Batch 4

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_hsv_color.png](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_hsv_color.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_hsv_color.png*

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_hsv_xy.png](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_hsv_xy.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_hsv_xy.png*

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_lab_color.png](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_lab_color.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_lab_color.png*

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_lab_xy.png](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_lab_xy.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_lab_xy.png*

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_rgb_color.png](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_rgb_color.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_rgb_color.png*

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_rgb_xy.png](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_rgb_xy.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k_grid_rgb_xy.png*

![Castle_Mountain_jpg_k_grid_hsv_color.png](../reports/figures/Castle_Mountain_jpg_k_grid_hsv_color.png)

*Castle_Mountain_jpg_k_grid_hsv_color.png*

![Castle_Mountain_jpg_k_grid_hsv_xy.png](../reports/figures/Castle_Mountain_jpg_k_grid_hsv_xy.png)

*Castle_Mountain_jpg_k_grid_hsv_xy.png*

![Castle_Mountain_jpg_k_grid_lab_color.png](../reports/figures/Castle_Mountain_jpg_k_grid_lab_color.png)

*Castle_Mountain_jpg_k_grid_lab_color.png*

![Castle_Mountain_jpg_k_grid_lab_xy.png](../reports/figures/Castle_Mountain_jpg_k_grid_lab_xy.png)

*Castle_Mountain_jpg_k_grid_lab_xy.png*

![Castle_Mountain_jpg_k_grid_rgb_color.png](../reports/figures/Castle_Mountain_jpg_k_grid_rgb_color.png)

*Castle_Mountain_jpg_k_grid_rgb_color.png*

![Castle_Mountain_jpg_k_grid_rgb_xy.png](../reports/figures/Castle_Mountain_jpg_k_grid_rgb_xy.png)

*Castle_Mountain_jpg_k_grid_rgb_xy.png*

### 11.5. K-Grid Batch 5

![Catoctin_Mountain_and_farm_MD1_jpg_k_grid_hsv_color.png](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k_grid_hsv_color.png)

*Catoctin_Mountain_and_farm_MD1_jpg_k_grid_hsv_color.png*

![Catoctin_Mountain_and_farm_MD1_jpg_k_grid_hsv_xy.png](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k_grid_hsv_xy.png)

*Catoctin_Mountain_and_farm_MD1_jpg_k_grid_hsv_xy.png*

![Catoctin_Mountain_and_farm_MD1_jpg_k_grid_lab_color.png](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k_grid_lab_color.png)

*Catoctin_Mountain_and_farm_MD1_jpg_k_grid_lab_color.png*

![Catoctin_Mountain_and_farm_MD1_jpg_k_grid_lab_xy.png](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k_grid_lab_xy.png)

*Catoctin_Mountain_and_farm_MD1_jpg_k_grid_lab_xy.png*

![Catoctin_Mountain_and_farm_MD1_jpg_k_grid_rgb_color.png](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k_grid_rgb_color.png)

*Catoctin_Mountain_and_farm_MD1_jpg_k_grid_rgb_color.png*

![Catoctin_Mountain_and_farm_MD1_jpg_k_grid_rgb_xy.png](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k_grid_rgb_xy.png)

*Catoctin_Mountain_and_farm_MD1_jpg_k_grid_rgb_xy.png*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_hsv_color.png](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_hsv_color.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_hsv_color.png*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_hsv_xy.png](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_hsv_xy.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_hsv_xy.png*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_lab_color.png](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_lab_color.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_lab_color.png*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_lab_xy.png](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_lab_xy.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_lab_xy.png*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_rgb_color.png](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_rgb_color.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_rgb_color.png*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_rgb_xy.png](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_rgb_xy.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k_grid_rgb_xy.png*

### 11.6. K-Grid Batch 6

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_hsv_color.png](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_hsv_color.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_hsv_color.png*

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_hsv_xy.png](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_hsv_xy.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_hsv_xy.png*

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_lab_color.png](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_lab_color.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_lab_color.png*

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_lab_xy.png](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_lab_xy.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_lab_xy.png*

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_rgb_color.png](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_rgb_color.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_rgb_color.png*

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_rgb_xy.png](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_rgb_xy.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k_grid_rgb_xy.png*

![jpg_k_grid_hsv_color.png](../reports/figures/jpg_k_grid_hsv_color.png)

*jpg_k_grid_hsv_color.png*

![jpg_k_grid_hsv_xy.png](../reports/figures/jpg_k_grid_hsv_xy.png)

*jpg_k_grid_hsv_xy.png*

![jpg_k_grid_lab_color.png](../reports/figures/jpg_k_grid_lab_color.png)

*jpg_k_grid_lab_color.png*

![jpg_k_grid_lab_xy.png](../reports/figures/jpg_k_grid_lab_xy.png)

*jpg_k_grid_lab_xy.png*

![jpg_k_grid_rgb_color.png](../reports/figures/jpg_k_grid_rgb_color.png)

*jpg_k_grid_rgb_color.png*

![jpg_k_grid_rgb_xy.png](../reports/figures/jpg_k_grid_rgb_xy.png)

*jpg_k_grid_rgb_xy.png*

### 11.7. K-Grid Batch 7

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_hsv_color.png](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_hsv_color.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_hsv_color.png*

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_hsv_xy.png](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_hsv_xy.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_hsv_xy.png*

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_lab_color.png](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_lab_color.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_lab_color.png*

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_lab_xy.png](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_lab_xy.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_lab_xy.png*

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_rgb_color.png](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_rgb_color.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_rgb_color.png*

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_rgb_xy.png](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_rgb_xy.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k_grid_rgb_xy.png*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_hsv_color.png](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_hsv_color.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_hsv_color.png*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_hsv_xy.png](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_hsv_xy.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_hsv_xy.png*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_lab_color.png](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_lab_color.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_lab_color.png*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_lab_xy.png](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_lab_xy.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_lab_xy.png*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_rgb_color.png](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_rgb_color.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_rgb_color.png*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_rgb_xy.png](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_rgb_xy.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k_grid_rgb_xy.png*

### 11.8. K-Grid Batch 8

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_hsv_color.png](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_hsv_color.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_hsv_color.png*

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_hsv_xy.png](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_hsv_xy.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_hsv_xy.png*

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_lab_color.png](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_lab_color.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_lab_color.png*

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_lab_xy.png](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_lab_xy.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_lab_xy.png*

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_rgb_color.png](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_rgb_color.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_rgb_color.png*

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_rgb_xy.png](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_rgb_xy.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k_grid_rgb_xy.png*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_hsv_color.png](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_hsv_color.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_hsv_color.png*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_hsv_xy.png](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_hsv_xy.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_hsv_xy.png*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_lab_color.png](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_lab_color.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_lab_color.png*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_lab_xy.png](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_lab_xy.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_lab_xy.png*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_rgb_color.png](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_rgb_color.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_rgb_color.png*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_rgb_xy.png](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_rgb_xy.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k_grid_rgb_xy.png*

### 11.9. K-Grid Batch 9

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_hsv_color.png](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_hsv_color.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_hsv_color.png*

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_hsv_xy.png](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_hsv_xy.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_hsv_xy.png*

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_lab_color.png](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_lab_color.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_lab_color.png*

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_lab_xy.png](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_lab_xy.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_lab_xy.png*

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_rgb_color.png](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_rgb_color.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_rgb_color.png*

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_rgb_xy.png](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_rgb_xy.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k_grid_rgb_xy.png*

![Sunset_by_Caspar_David_Friedrich_jpg_k_grid_hsv_color.png](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k_grid_hsv_color.png)

*Sunset_by_Caspar_David_Friedrich_jpg_k_grid_hsv_color.png*

![Sunset_by_Caspar_David_Friedrich_jpg_k_grid_hsv_xy.png](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k_grid_hsv_xy.png)

*Sunset_by_Caspar_David_Friedrich_jpg_k_grid_hsv_xy.png*

![Sunset_by_Caspar_David_Friedrich_jpg_k_grid_lab_color.png](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k_grid_lab_color.png)

*Sunset_by_Caspar_David_Friedrich_jpg_k_grid_lab_color.png*

![Sunset_by_Caspar_David_Friedrich_jpg_k_grid_lab_xy.png](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k_grid_lab_xy.png)

*Sunset_by_Caspar_David_Friedrich_jpg_k_grid_lab_xy.png*

![Sunset_by_Caspar_David_Friedrich_jpg_k_grid_rgb_color.png](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k_grid_rgb_color.png)

*Sunset_by_Caspar_David_Friedrich_jpg_k_grid_rgb_color.png*

![Sunset_by_Caspar_David_Friedrich_jpg_k_grid_rgb_xy.png](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k_grid_rgb_xy.png)

*Sunset_by_Caspar_David_Friedrich_jpg_k_grid_rgb_xy.png*

### 11.10. K-Grid Batch 10

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_hsv_color.png](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_hsv_color.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_hsv_color.png*

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_hsv_xy.png](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_hsv_xy.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_hsv_xy.png*

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_lab_color.png](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_lab_color.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_lab_color.png*

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_lab_xy.png](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_lab_xy.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_lab_xy.png*

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_rgb_color.png](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_rgb_color.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_rgb_color.png*

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_rgb_xy.png](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_rgb_xy.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k_grid_rgb_xy.png*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_hsv_color.png](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_hsv_color.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_hsv_color.png*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_hsv_xy.png](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_hsv_xy.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_hsv_xy.png*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_lab_color.png](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_lab_color.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_lab_color.png*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_lab_xy.png](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_lab_xy.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_lab_xy.png*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_rgb_color.png](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_rgb_color.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_rgb_color.png*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_rgb_xy.png](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_rgb_xy.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k_grid_rgb_xy.png*

## 12. Tất Cả Hình Ảnh Model Sau Khi Train - Output Image Gallery

Phần appendix dưới đây nhúng trực tiếp toàn bộ ảnh comparison đã train. Mỗi ảnh là một cấu hình model cụ thể gồm `image_id`, `K`, `color_space`, và `use_xy`.

Ghi chú: số lượng ảnh lớn vì workflow đã train đủ `20 × 9 × 3 × 2 = 1080` cấu hình.

### 12.1. Trained Output Batch 1

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_hsv_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_hsv_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_lab_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_lab_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=lab | use_xy=True*

### 12.2. Trained Output Batch 2

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_rgb_color_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k3_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k4_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k5_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k6_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k7_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k8_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k9_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k10_rgb_xy_comparison.png)

*image_id=004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_hsv_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_hsv_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=hsv | use_xy=True*

### 12.3. Trained Output Batch 3

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_lab_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_lab_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=lab | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_rgb_color_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k3_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k4_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k5_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k6_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k7_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k8_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k9_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k10_rgb_xy_comparison.png)

*image_id=Algoma_Gabrielle_Rock_I0012362_jpg | k=10 | color_space=rgb | use_xy=True*

### 12.4. Trained Output Batch 4

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_hsv_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_hsv_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_lab_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_lab_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=lab | use_xy=True*

### 12.5. Trained Output Batch 5

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_rgb_color_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k3_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k4_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k5_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k6_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k7_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k8_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k9_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k10_rgb_xy_comparison.png)

*image_id=Barn_on_Mastl_mountain_Gherd_ina_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_hsv_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_hsv_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=hsv | use_xy=True*

### 12.6. Trained Output Batch 6

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_lab_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_lab_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=lab | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_rgb_color_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k3_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k4_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k5_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k6_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k7_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k8_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k9_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k10_rgb_xy_comparison.png)

*image_id=Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | k=10 | color_space=rgb | use_xy=True*

### 12.7. Trained Output Batch 7

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_hsv_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_hsv_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_lab_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_lab_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=lab | use_xy=True*

### 12.8. Trained Output Batch 8

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_rgb_color_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k2_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k4_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k5_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k6_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k7_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k8_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k9_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k10_rgb_xy_comparison.png)

*image_id=Bontecou_Lake_Milky_Way_panorama_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_hsv_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_hsv_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=hsv | use_xy=True*

### 12.9. Trained Output Batch 9

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_lab_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_lab_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=lab | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_rgb_color_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k3_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k4_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k5_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k6_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k7_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k8_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k9_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k10_rgb_xy_comparison.png)

*image_id=C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | k=10 | color_space=rgb | use_xy=True*

### 12.10. Trained Output Batch 10

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=hsv | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_hsv_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=hsv | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=hsv | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_hsv_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=hsv | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=lab | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_lab_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=lab | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=lab | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=lab | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_lab_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=lab | use_xy=True*

### 12.11. Trained Output Batch 11

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=rgb | use_xy=False](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_rgb_color_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=rgb | use_xy=False*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=2 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k3_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=3 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k4_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=4 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k5_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=5 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k6_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=6 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k7_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=7 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k8_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=8 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k9_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=9 | color_space=rgb | use_xy=True*

![image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=rgb | use_xy=True](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k10_rgb_xy_comparison.png)

*image_id=Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | k=10 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k2_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k3_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k4_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k5_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k6_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k7_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k8_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k9_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Castle_Mountain_jpg_k10_hsv_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Castle_Mountain_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k2_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k3_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k4_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k5_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k6_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k7_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k8_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k9_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Castle_Mountain_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Castle_Mountain_jpg_k10_hsv_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=10 | color_space=hsv | use_xy=True*

### 12.12. Trained Output Batch 12

![image_id=Castle_Mountain_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k2_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k3_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k4_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k5_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k6_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k7_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k8_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k9_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Castle_Mountain_jpg_k10_lab_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Castle_Mountain_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k2_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k3_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k4_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k5_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k6_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k7_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k8_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k9_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Castle_Mountain_jpg_k10_lab_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=10 | color_space=lab | use_xy=True*

![image_id=Castle_Mountain_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k2_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k3_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k4_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k5_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k6_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k7_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k8_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k9_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Castle_Mountain_jpg_k10_rgb_color_comparison.png)

*image_id=Castle_Mountain_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Castle_Mountain_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k2_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k3_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k4_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k5_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k6_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k7_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k8_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k9_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Castle_Mountain_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Castle_Mountain_jpg_k10_rgb_xy_comparison.png)

*image_id=Castle_Mountain_jpg | k=10 | color_space=rgb | use_xy=True*

### 12.13. Trained Output Batch 13

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_hsv_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_hsv_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_lab_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_lab_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=lab | use_xy=True*

### 12.14. Trained Output Batch 14

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_rgb_color_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k3_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k4_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k5_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k6_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k7_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k8_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k9_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k10_rgb_xy_comparison.png)

*image_id=Catoctin_Mountain_and_farm_MD1_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_hsv_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_hsv_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=hsv | use_xy=True*

### 12.15. Trained Output Batch 15

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_lab_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_lab_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=lab | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_rgb_color_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k3_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k4_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k5_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k6_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k7_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k8_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k9_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k10_rgb_xy_comparison.png)

*image_id=Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | k=10 | color_space=rgb | use_xy=True*

### 12.16. Trained Output Batch 16

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_hsv_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_hsv_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_lab_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_lab_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=lab | use_xy=True*

### 12.17. Trained Output Batch 17

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_rgb_color_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k3_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k4_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k5_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k6_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k7_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k8_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k9_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k10_rgb_xy_comparison.png)

*image_id=Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=hsv | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_hsv_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=hsv | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=hsv | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=hsv | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_hsv_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=hsv | use_xy=True*

### 12.18. Trained Output Batch 18

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=lab | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_lab_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=lab | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=lab | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_lab_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=lab | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=rgb | use_xy=False](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_rgb_color_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=rgb | use_xy=False*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=2 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k3_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=3 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k4_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=4 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k5_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=5 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k6_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=6 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k7_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=7 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k8_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=8 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k9_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=9 | color_space=rgb | use_xy=True*

![image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=rgb | use_xy=True](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k10_rgb_xy_comparison.png)

*image_id=Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | k=10 | color_space=rgb | use_xy=True*

### 12.19. Trained Output Batch 19

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_hsv_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_hsv_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_lab_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_lab_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=lab | use_xy=True*

### 12.20. Trained Output Batch 20

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_rgb_color_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k2_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k4_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k5_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k6_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k7_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k8_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k9_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k10_rgb_xy_comparison.png)

*image_id=Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=hsv | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_hsv_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=hsv | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=hsv | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=hsv | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_hsv_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=hsv | use_xy=True*

### 12.21. Trained Output Batch 21

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=lab | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_lab_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=lab | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=lab | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_lab_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=lab | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=rgb | use_xy=False](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_rgb_color_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=rgb | use_xy=False*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=2 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k3_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=3 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k4_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=4 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k5_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=5 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k6_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=6 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k7_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=7 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k8_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=8 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k9_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=9 | color_space=rgb | use_xy=True*

![image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=rgb | use_xy=True](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k10_rgb_xy_comparison.png)

*image_id=Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | k=10 | color_space=rgb | use_xy=True*

### 12.22. Trained Output Batch 22

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_hsv_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_hsv_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_lab_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_lab_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=lab | use_xy=True*

### 12.23. Trained Output Batch 23

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_rgb_color_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k2_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k4_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k5_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k6_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k7_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k8_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k9_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k10_rgb_xy_comparison.png)

*image_id=Ruisseau_du_Vialais_-_March_2021_-_B_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_hsv_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_hsv_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=hsv | use_xy=True*

### 12.24. Trained Output Batch 24

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_lab_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_lab_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=lab | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_rgb_color_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k3_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k4_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k5_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k6_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k7_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k8_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k9_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k10_rgb_xy_comparison.png)

*image_id=Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | k=10 | color_space=rgb | use_xy=True*

### 12.25. Trained Output Batch 25

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_hsv_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_hsv_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=hsv | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_lab_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=lab | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=lab | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_lab_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=lab | use_xy=True*

### 12.26. Trained Output Batch 26

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_rgb_color_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k3_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k4_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k5_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k6_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k7_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k8_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k9_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k10_rgb_xy_comparison.png)

*image_id=Sunset_by_Caspar_David_Friedrich_jpg | k=10 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=hsv | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_hsv_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=hsv | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=hsv | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=hsv | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_hsv_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=hsv | use_xy=True*

### 12.27. Trained Output Batch 27

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=lab | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_lab_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=lab | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=lab | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_lab_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=lab | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=rgb | use_xy=False](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_rgb_color_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=rgb | use_xy=False*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k2_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=2 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=3 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k4_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=4 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k5_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=5 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k6_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=6 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k7_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=7 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k8_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=8 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k9_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=9 | color_space=rgb | use_xy=True*

![image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=rgb | use_xy=True](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k10_rgb_xy_comparison.png)

*image_id=Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | k=10 | color_space=rgb | use_xy=True*

### 12.28. Trained Output Batch 28

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=hsv | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_hsv_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=hsv | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=hsv | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_hsv_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=hsv | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=lab | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_lab_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=lab | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=lab | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=lab | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_lab_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=lab | use_xy=True*

### 12.29. Trained Output Batch 29

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=rgb | use_xy=False](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_rgb_color_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=rgb | use_xy=False*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=2 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k3_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=3 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k4_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=4 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k5_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=5 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k6_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=6 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k7_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=7 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k8_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=8 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k9_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=9 | color_space=rgb | use_xy=True*

![image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=rgb | use_xy=True](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k10_rgb_xy_comparison.png)

*image_id=Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | k=10 | color_space=rgb | use_xy=True*

![image_id=jpg | k=2 | color_space=hsv | use_xy=False](../reports/figures/jpg_k2_hsv_color_comparison.png)

*image_id=jpg | k=2 | color_space=hsv | use_xy=False*

![image_id=jpg | k=3 | color_space=hsv | use_xy=False](../reports/figures/jpg_k3_hsv_color_comparison.png)

*image_id=jpg | k=3 | color_space=hsv | use_xy=False*

![image_id=jpg | k=4 | color_space=hsv | use_xy=False](../reports/figures/jpg_k4_hsv_color_comparison.png)

*image_id=jpg | k=4 | color_space=hsv | use_xy=False*

![image_id=jpg | k=5 | color_space=hsv | use_xy=False](../reports/figures/jpg_k5_hsv_color_comparison.png)

*image_id=jpg | k=5 | color_space=hsv | use_xy=False*

![image_id=jpg | k=6 | color_space=hsv | use_xy=False](../reports/figures/jpg_k6_hsv_color_comparison.png)

*image_id=jpg | k=6 | color_space=hsv | use_xy=False*

![image_id=jpg | k=7 | color_space=hsv | use_xy=False](../reports/figures/jpg_k7_hsv_color_comparison.png)

*image_id=jpg | k=7 | color_space=hsv | use_xy=False*

![image_id=jpg | k=8 | color_space=hsv | use_xy=False](../reports/figures/jpg_k8_hsv_color_comparison.png)

*image_id=jpg | k=8 | color_space=hsv | use_xy=False*

![image_id=jpg | k=9 | color_space=hsv | use_xy=False](../reports/figures/jpg_k9_hsv_color_comparison.png)

*image_id=jpg | k=9 | color_space=hsv | use_xy=False*

![image_id=jpg | k=10 | color_space=hsv | use_xy=False](../reports/figures/jpg_k10_hsv_color_comparison.png)

*image_id=jpg | k=10 | color_space=hsv | use_xy=False*

![image_id=jpg | k=2 | color_space=hsv | use_xy=True](../reports/figures/jpg_k2_hsv_xy_comparison.png)

*image_id=jpg | k=2 | color_space=hsv | use_xy=True*

![image_id=jpg | k=3 | color_space=hsv | use_xy=True](../reports/figures/jpg_k3_hsv_xy_comparison.png)

*image_id=jpg | k=3 | color_space=hsv | use_xy=True*

![image_id=jpg | k=4 | color_space=hsv | use_xy=True](../reports/figures/jpg_k4_hsv_xy_comparison.png)

*image_id=jpg | k=4 | color_space=hsv | use_xy=True*

![image_id=jpg | k=5 | color_space=hsv | use_xy=True](../reports/figures/jpg_k5_hsv_xy_comparison.png)

*image_id=jpg | k=5 | color_space=hsv | use_xy=True*

![image_id=jpg | k=6 | color_space=hsv | use_xy=True](../reports/figures/jpg_k6_hsv_xy_comparison.png)

*image_id=jpg | k=6 | color_space=hsv | use_xy=True*

![image_id=jpg | k=7 | color_space=hsv | use_xy=True](../reports/figures/jpg_k7_hsv_xy_comparison.png)

*image_id=jpg | k=7 | color_space=hsv | use_xy=True*

![image_id=jpg | k=8 | color_space=hsv | use_xy=True](../reports/figures/jpg_k8_hsv_xy_comparison.png)

*image_id=jpg | k=8 | color_space=hsv | use_xy=True*

![image_id=jpg | k=9 | color_space=hsv | use_xy=True](../reports/figures/jpg_k9_hsv_xy_comparison.png)

*image_id=jpg | k=9 | color_space=hsv | use_xy=True*

![image_id=jpg | k=10 | color_space=hsv | use_xy=True](../reports/figures/jpg_k10_hsv_xy_comparison.png)

*image_id=jpg | k=10 | color_space=hsv | use_xy=True*

### 12.30. Trained Output Batch 30

![image_id=jpg | k=2 | color_space=lab | use_xy=False](../reports/figures/jpg_k2_lab_color_comparison.png)

*image_id=jpg | k=2 | color_space=lab | use_xy=False*

![image_id=jpg | k=3 | color_space=lab | use_xy=False](../reports/figures/jpg_k3_lab_color_comparison.png)

*image_id=jpg | k=3 | color_space=lab | use_xy=False*

![image_id=jpg | k=4 | color_space=lab | use_xy=False](../reports/figures/jpg_k4_lab_color_comparison.png)

*image_id=jpg | k=4 | color_space=lab | use_xy=False*

![image_id=jpg | k=5 | color_space=lab | use_xy=False](../reports/figures/jpg_k5_lab_color_comparison.png)

*image_id=jpg | k=5 | color_space=lab | use_xy=False*

![image_id=jpg | k=6 | color_space=lab | use_xy=False](../reports/figures/jpg_k6_lab_color_comparison.png)

*image_id=jpg | k=6 | color_space=lab | use_xy=False*

![image_id=jpg | k=7 | color_space=lab | use_xy=False](../reports/figures/jpg_k7_lab_color_comparison.png)

*image_id=jpg | k=7 | color_space=lab | use_xy=False*

![image_id=jpg | k=8 | color_space=lab | use_xy=False](../reports/figures/jpg_k8_lab_color_comparison.png)

*image_id=jpg | k=8 | color_space=lab | use_xy=False*

![image_id=jpg | k=9 | color_space=lab | use_xy=False](../reports/figures/jpg_k9_lab_color_comparison.png)

*image_id=jpg | k=9 | color_space=lab | use_xy=False*

![image_id=jpg | k=10 | color_space=lab | use_xy=False](../reports/figures/jpg_k10_lab_color_comparison.png)

*image_id=jpg | k=10 | color_space=lab | use_xy=False*

![image_id=jpg | k=2 | color_space=lab | use_xy=True](../reports/figures/jpg_k2_lab_xy_comparison.png)

*image_id=jpg | k=2 | color_space=lab | use_xy=True*

![image_id=jpg | k=3 | color_space=lab | use_xy=True](../reports/figures/jpg_k3_lab_xy_comparison.png)

*image_id=jpg | k=3 | color_space=lab | use_xy=True*

![image_id=jpg | k=4 | color_space=lab | use_xy=True](../reports/figures/jpg_k4_lab_xy_comparison.png)

*image_id=jpg | k=4 | color_space=lab | use_xy=True*

![image_id=jpg | k=5 | color_space=lab | use_xy=True](../reports/figures/jpg_k5_lab_xy_comparison.png)

*image_id=jpg | k=5 | color_space=lab | use_xy=True*

![image_id=jpg | k=6 | color_space=lab | use_xy=True](../reports/figures/jpg_k6_lab_xy_comparison.png)

*image_id=jpg | k=6 | color_space=lab | use_xy=True*

![image_id=jpg | k=7 | color_space=lab | use_xy=True](../reports/figures/jpg_k7_lab_xy_comparison.png)

*image_id=jpg | k=7 | color_space=lab | use_xy=True*

![image_id=jpg | k=8 | color_space=lab | use_xy=True](../reports/figures/jpg_k8_lab_xy_comparison.png)

*image_id=jpg | k=8 | color_space=lab | use_xy=True*

![image_id=jpg | k=9 | color_space=lab | use_xy=True](../reports/figures/jpg_k9_lab_xy_comparison.png)

*image_id=jpg | k=9 | color_space=lab | use_xy=True*

![image_id=jpg | k=10 | color_space=lab | use_xy=True](../reports/figures/jpg_k10_lab_xy_comparison.png)

*image_id=jpg | k=10 | color_space=lab | use_xy=True*

![image_id=jpg | k=2 | color_space=rgb | use_xy=False](../reports/figures/jpg_k2_rgb_color_comparison.png)

*image_id=jpg | k=2 | color_space=rgb | use_xy=False*

![image_id=jpg | k=3 | color_space=rgb | use_xy=False](../reports/figures/jpg_k3_rgb_color_comparison.png)

*image_id=jpg | k=3 | color_space=rgb | use_xy=False*

![image_id=jpg | k=4 | color_space=rgb | use_xy=False](../reports/figures/jpg_k4_rgb_color_comparison.png)

*image_id=jpg | k=4 | color_space=rgb | use_xy=False*

![image_id=jpg | k=5 | color_space=rgb | use_xy=False](../reports/figures/jpg_k5_rgb_color_comparison.png)

*image_id=jpg | k=5 | color_space=rgb | use_xy=False*

![image_id=jpg | k=6 | color_space=rgb | use_xy=False](../reports/figures/jpg_k6_rgb_color_comparison.png)

*image_id=jpg | k=6 | color_space=rgb | use_xy=False*

![image_id=jpg | k=7 | color_space=rgb | use_xy=False](../reports/figures/jpg_k7_rgb_color_comparison.png)

*image_id=jpg | k=7 | color_space=rgb | use_xy=False*

![image_id=jpg | k=8 | color_space=rgb | use_xy=False](../reports/figures/jpg_k8_rgb_color_comparison.png)

*image_id=jpg | k=8 | color_space=rgb | use_xy=False*

![image_id=jpg | k=9 | color_space=rgb | use_xy=False](../reports/figures/jpg_k9_rgb_color_comparison.png)

*image_id=jpg | k=9 | color_space=rgb | use_xy=False*

![image_id=jpg | k=10 | color_space=rgb | use_xy=False](../reports/figures/jpg_k10_rgb_color_comparison.png)

*image_id=jpg | k=10 | color_space=rgb | use_xy=False*

![image_id=jpg | k=2 | color_space=rgb | use_xy=True](../reports/figures/jpg_k2_rgb_xy_comparison.png)

*image_id=jpg | k=2 | color_space=rgb | use_xy=True*

![image_id=jpg | k=3 | color_space=rgb | use_xy=True](../reports/figures/jpg_k3_rgb_xy_comparison.png)

*image_id=jpg | k=3 | color_space=rgb | use_xy=True*

![image_id=jpg | k=4 | color_space=rgb | use_xy=True](../reports/figures/jpg_k4_rgb_xy_comparison.png)

*image_id=jpg | k=4 | color_space=rgb | use_xy=True*

![image_id=jpg | k=5 | color_space=rgb | use_xy=True](../reports/figures/jpg_k5_rgb_xy_comparison.png)

*image_id=jpg | k=5 | color_space=rgb | use_xy=True*

![image_id=jpg | k=6 | color_space=rgb | use_xy=True](../reports/figures/jpg_k6_rgb_xy_comparison.png)

*image_id=jpg | k=6 | color_space=rgb | use_xy=True*

![image_id=jpg | k=7 | color_space=rgb | use_xy=True](../reports/figures/jpg_k7_rgb_xy_comparison.png)

*image_id=jpg | k=7 | color_space=rgb | use_xy=True*

![image_id=jpg | k=8 | color_space=rgb | use_xy=True](../reports/figures/jpg_k8_rgb_xy_comparison.png)

*image_id=jpg | k=8 | color_space=rgb | use_xy=True*

![image_id=jpg | k=9 | color_space=rgb | use_xy=True](../reports/figures/jpg_k9_rgb_xy_comparison.png)

*image_id=jpg | k=9 | color_space=rgb | use_xy=True*

![image_id=jpg | k=10 | color_space=rgb | use_xy=True](../reports/figures/jpg_k10_rgb_xy_comparison.png)

*image_id=jpg | k=10 | color_space=rgb | use_xy=True*

## 13. Final Review

Kết quả review cuối:

| Check | Result |
|---|---|
| review status | passed |
| clean images | 20 |
| model runs | 1080 |
| expected model runs | 1080 |
| notebook code cells | 0 |
| assignment alignment | The source implements unsupervised K-Means Image Segmentation with landscape images. |

Notebook đã được review để đảm bảo không có code cell, không có lỗi font do encoding, và các ảnh output sau train được nhúng bằng Markdown image syntax chuẩn.